In [27]:
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import AgglomerativeClustering

In [28]:
# -------------------------------
# Project Settings
# -------------------------------

DATA_PATH = "raw_wholesale_customers.csv"
OUTPUT_PATH = "segmented_wholesale_customers.csv"

spending_features = [
    "Fresh",
    "Milk",
    "Grocery",
    "Frozen",
    "Detergents_Paper",
    "Delicassen"
]

k_clusters = 3
seed = 42

# -------------------------------
# Load Dataset
# -------------------------------

customers = pd.read_csv(DATA_PATH)
print()

print(customers.head())

print()
print("Dataset Information")
print(customers.info())


   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    i

In [29]:
# -------------------------------
# Select Spending Columns
# -------------------------------

X = customers[spending_features].copy()

print("Selected Features")
print(X.head())

Selected Features
   Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669  9656     7561     214              2674        1338
1   7057  9810     9568    1762              3293        1776
2   6353  8808     7684    2405              3516        7844
3  13265  1196     4221    6404               507        1788
4  22615  5410     7198    3915              1777        5185


In [30]:
# -------------------------------
# Handle Outliers using IQR Capping
# -------------------------------

def cap_outliers(column):

    q1 = column.quantile(0.25)
    q3 = column.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    return column.clip(lower=lower, upper=upper)

for feature in spending_features:
    X[feature] = cap_outliers(X[feature])

customers[spending_features] = X

print("Outlier capping completed.")
print()
print(X.describe())

Outlier capping completed.

              Fresh          Milk      Grocery       Frozen  Detergents_Paper  \
count    440.000000    440.000000    440.00000   440.000000        440.000000   
mean   11357.568182   5048.592045   7236.37500  2507.085795       2392.616477   
std    10211.542235   4386.377073   6596.53308  2408.297738       2940.794090   
min        3.000000     55.000000      3.00000    25.000000          3.000000   
25%     3127.750000   1533.000000   2153.00000   742.250000        256.750000   
50%     8504.000000   3627.000000   4755.50000  1526.000000        816.500000   
75%    16933.750000   7190.250000  10655.75000  3554.250000       3922.000000   
max    37642.750000  15676.125000  23409.87500  7772.250000       9419.875000   

        Delicassen  
count   440.000000  
mean   1266.715341  
std    1083.069792  
min       3.000000  
25%     408.250000  
50%     965.500000  
75%    1820.250000  
max    3938.250000  


In [31]:
# -------------------------------
# Scale Features
# -------------------------------

scaler = StandardScaler()

scaled_data = scaler.fit_transform(X)

print("Scaling Finished")
print("Shape:", scaled_data.shape)

Scaling Finished
Shape: (440, 6)


In [32]:
# -------------------------------
# Elbow Method
# -------------------------------

print("Elbow Method Results")
print("-" * 30)

sse = []

for k in range(1, 11):

    model = KMeans(
        n_clusters=k,
        random_state=seed,
        n_init="auto"
    )

    model.fit(scaled_data)

    sse.append(model.inertia_)

    print(f"k = {k}   SSE = {model.inertia_:.2f}")

Elbow Method Results
------------------------------
k = 1   SSE = 2640.00
k = 2   SSE = 1728.19
k = 3   SSE = 1363.45
k = 4   SSE = 1202.41
k = 5   SSE = 1070.15
k = 6   SSE = 964.76
k = 7   SSE = 921.14
k = 8   SSE = 776.63
k = 9   SSE = 726.88
k = 10   SSE = 707.41


In [33]:
# -------------------------------
# Train K-Means
# -------------------------------


kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

customers["Cluster"] = kmeans.fit_predict(scaled_data)

print()

print(customers.head())


   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  
0     1338.00        1  
1     1776.00        0  
2     3938.25        2  
3     1788.00        2  
4     3938.25        2  


In [34]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# -------------------------------
# Evaluate K-Means
# -------------------------------

kmeans_silhouette = silhouette_score(scaled_data, customers["Cluster"])
kmeans_dbi = davies_bouldin_score(scaled_data, customers["Cluster"])

print("K-Means Performance")
print("-" * 30)
print(f"Silhouette Score      : {kmeans_silhouette:.3f}")
print(f"Davies-Bouldin Index : {kmeans_dbi:.3f}")

K-Means Performance
------------------------------
Silhouette Score      : 0.340
Davies-Bouldin Index : 1.297


In [35]:
# -------------------------------
# Cluster Centers
# -------------------------------

original_centers = scaler.inverse_transform(kmeans.cluster_centers_)

centers = pd.DataFrame(
    original_centers,
    columns=spending_features
)

centers.index.name = "Cluster"

print("Cluster Centers")
print()
print(centers.round(2))

Cluster Centers

            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         5869.23  10142.25  16653.32  1355.63           7022.05     1482.60
1         9418.37   2685.47   3583.98  2052.33            942.49      750.71
2        22881.83   5870.35   6773.44  5057.43           1209.38     2452.04


In [36]:

# -------------------------------
# Second Clustering Algorithm
# -------------------------------

agglomerative = AgglomerativeClustering(
    n_clusters=3
)

customers["Agglomerative_Cluster"] = agglomerative.fit_predict(scaled_data)

print("Agglomerative Clustering Finished")
print()

print(customers.head())

Agglomerative Clustering Finished

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  Agglomerative_Cluster  
0     1338.00        1                      2  
1     1776.00        0                      2  
2     3938.25        2                      0  
3     1788.00        2                      0  
4     3938.25        2                      0  


In [37]:
# -------------------------------
# Compare Clustering Methods
# -------------------------------

agg_silhouette = silhouette_score(
    scaled_data,
    customers["Agglomerative_Cluster"]
)

print("Model Comparison")
print("-" * 30)

print(f"K-Means Silhouette Score       : {kmeans_silhouette:.3f}")
print(f"Agglomerative Silhouette Score : {agg_silhouette:.3f}")

print()

if kmeans_silhouette > agg_silhouette:
    print("K-Means created better-separated clusters.")
elif agg_silhouette > kmeans_silhouette:
    print("Agglomerative Clustering created better-separated clusters.")
else:
    print("Both methods produced similar clustering quality.")
    

Model Comparison
------------------------------
K-Means Silhouette Score       : 0.340
Agglomerative Silhouette Score : 0.284

K-Means created better-separated clusters.


In [38]:
# -------------------------------
# Compare Cluster Labels
# -------------------------------

sample_customers = customers.loc[
    [0, 1, 2],
    [
        "Channel",
        "Region",
        "Fresh",
        "Milk",
        "Grocery",
        "Frozen",
        "Detergents_Paper",
        "Delicassen",
        "Cluster",
        "Agglomerative_Cluster"
    ]
]

print("Three Sample Customers")
print()
print(sample_customers)

Three Sample Customers

   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   

   Delicassen  Cluster  Agglomerative_Cluster  
0     1338.00        1                      2  
1     1776.00        0                      2  
2     3938.25        2                      0  


In [39]:
# -------------------------------
# Save Final Dataset
# -------------------------------

customers.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Dataset saved successfully.")
print(f"Output file: {OUTPUT_PATH}")

Dataset saved successfully.
Output file: segmented_wholesale_customers.csv
